# Customer Churn Prediction
**Dataset:** Telco Customer Churn (IBM / Kaggle — blastchar)  
**Goal:** Build a supervised ML classification model to predict which telecom customers are likely to churn  
**Author:** Muhammad Umer  

---

## Project Overview

Customer churn — the rate at which customers stop doing business with a company — is one of the most expensive problems in subscription-based industries. In telecommunications, acquiring a new customer costs 5 to 25 times more than retaining an existing one. Identifying customers who are *about to leave* before they actually do gives the business a window to intervene with targeted retention offers.

This notebook builds an end-to-end supervised machine learning pipeline to solve that problem. The dataset contains information on 7,043 telecom customers including their demographics, account details, subscribed services, and whether they churned.

**The workflow follows seven stages:**

1. Exploratory Data Analysis — understand the data, the churn rate, and which features visually separate churners from non-churners
2. Data Cleaning — fix dtype issues, handle missing values, drop non-predictive columns
3. Feature Engineering — create three new features that capture information not directly present in the raw columns
4. Preprocessing Pipeline — build a reusable `ColumnTransformer` inside a `Pipeline` to prevent data leakage during model training
5. Baseline Model Comparison — compare three classification models using stratified cross-validation and ROC-AUC
6. Hyperparameter Tuning — tune the best model using `GridSearchCV`
7. Final Evaluation & Business Recommendations — evaluate on the held-out test set and translate findings into actionable retention strategy

**Tools:** Python 3.13 · pandas · matplotlib · seaborn · scikit-learn 1.8.0

## Step 0 — Setup and Data Loading

Before any analysis begins, we load all libraries the project will use and take a first look at the raw data to understand its structure, column types, and the target variable distribution.

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import os               # File path management — used to create the Figures folder
import warnings
warnings.filterwarnings('ignore')   # Suppress convergence and deprecation warnings
                                    # for cleaner notebook output

# ── Data manipulation ─────────────────────────────────────────────────────────
import numpy as np      
import pandas as pd     

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt     
import matplotlib.ticker as mticker # Fine-grained axis formatting (%, comma separators)
import seaborn as sns               

# ── Scikit-learn: Data splitting & validation ─────────────────────────────────
from sklearn.model_selection import (
    train_test_split,   # Splits data into train and test sets — done once, at the start
    StratifiedKFold,    # K-fold cross-validation that preserves class proportions
                        # Essential here because churn (~26%) is an imbalanced class —
                        # regular KFold can produce folds with very few churners
    cross_val_score,    # Runs CV on a model and returns per-fold scores
    GridSearchCV        # Exhaustive hyperparameter search with built-in cross-validation
)

# ── Scikit-learn: Preprocessing ───────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, OneHotEncoder
# StandardScaler  — normalises numerical features to mean=0, std=1
#                   Required for Logistic Regression and KNN which are distance-sensitive
# OneHotEncoder   — converts categorical columns (e.g. Contract type) into binary columns
#                   sklearn's version handles unknown categories gracefully

from sklearn.impute import SimpleImputer
# Fills missing values — we will discover one column with blanks in Step 2

from sklearn.compose import ColumnTransformer
# Applies different preprocessing steps to different column subsets simultaneously
# e.g. scale numerical columns AND one-hot encode categorical columns in one step

from sklearn.pipeline import Pipeline
# Chains preprocessing + model into a single object
# This is the production-ready approach — it prevents data leakage because
# the scaler never sees test data during cross-validation or GridSearch

# ── Scikit-learn: Models ──────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
# Interpretable linear classifier — strong baseline for binary classification
# Outputs probabilities, which is essential for ROC-AUC scoring

from sklearn.ensemble import RandomForestClassifier
# Tree-based ensemble — handles non-linear relationships and interactions between features
# Also provides feature importances directly (Step 8)

from sklearn.neighbors import KNeighborsClassifier
# Distance-based classifier — included as a third comparison point
# Sensitive to feature scale, which is why scaling inside the pipeline matters

# ── Scikit-learn: Evaluation metrics ─────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,         # % of correct predictions — useful context but misleading alone
    roc_auc_score,          # Primary metric: measures how well the model separates the two
                            # classes regardless of threshold. 0.5 = random, 1.0 = perfect
    precision_score,        # Of all predicted churners, how many actually churned?
    recall_score,           # Of all actual churners, how many did we catch?
                            # Recall is usually more important in churn — missing a churner
                            # costs more than a false alarm
    confusion_matrix,       # 2x2 table of TP, FP, FN, TN — visualised as heatmap
    classification_report   # Full per-class breakdown of precision, recall, F1
)

# ── Global plot settings ──────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.family'] = 'Arial'   # Avoids font-related rendering issues on Windows
plt.rcParams['figure.dpi'] = 120

# ── Output folder ─────────────────────────────────────────────────────────────
FIGURES_DIR = 'Figures'
os.makedirs(FIGURES_DIR, exist_ok=True)  # Creates folder if it doesn't exist yet

print('All libraries imported successfully.')
print(f'scikit-learn version: {__import__("sklearn").__version__}')

All libraries imported successfully.
scikit-learn version: 1.8.0


---
### Load and Inspect the Raw Data

We load the CSV and immediately check three things:
- **Shape** — how many rows and columns do we have?
- **Data types** — are all columns stored as the correct type? (Spoiler: one is not.)
- **Target variable** — what is the churn rate and is the dataset imbalanced?

These three checks determine everything that comes next in Steps 1 and 2.

In [4]:
# ── Load the CSV ──────────────────────────────────────────────────────────────
CSV_PATH = os.path.join('Data', 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = pd.read_csv(CSV_PATH)

# ── Shape and column names ────────────────────────────────────────────────────
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
print('Columns:')
print(df.columns.tolist())

Shape: 7,043 rows × 21 columns

Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [ ]:
# ── Data types ────────────────────────────────────────────────────────────────
# We check dtypes before anything else — dtype errors silently break downstream
# operations (e.g. a numeric column stored as object cannot be scaled)
print('--- Data Types ---')
print(df.dtypes)

--- Data Types ---
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object



In [5]:
# ── Missing values ────────────────────────────────────────────────────────────
print('--- Missing Values ---')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No null values detected.')

--- Missing Values ---
No null values detected.


In [7]:
# ── Target variable distribution ──────────────────────────────────────────────
# Class imbalance affects which CV strategy and metrics we use throughout the project
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

print('--- Target Variable: Churn ---')
print(pd.DataFrame({'Count': churn_counts, 'Percentage': churn_pct.round(1)}))

--- Target Variable: Churn ---
       Count  Percentage
Churn                   
No      5174        73.5
Yes     1869        26.5


In [8]:
# ── First look at the data ────────────────────────────────────────────────────
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Step 0 Findings

The dataset contains 7,043 customers across 21 columns. 

Three immediate observations from the first inspection:

- **TotalCharges is stored as string** despite being a numeric billing column. This is because a small number of rows contain blank strings instead of a numeric value. This will be converted to float and the blank rows handled in Step 2.
- **SeniorCitizen is stored as int64** (0/1) rather than Yes/No like the other binary columns. We will treat it as categorical during preprocessing to keep it consistent with the rest of the binary features.
- **The dataset is moderately imbalanced** — 73.5% of customers did not churn versus 26.5% who did. This rules out accuracy as a reliable evaluation metric and confirms that StratifiedKFold and ROC-AUC are the correct choices for model comparison in Steps 5 and 6.